In [15]:
import nflreadpy as nfl
import polars as pl
import sys
from pathlib import Path
import os
import plotly.express as px
import plotly.graph_objects as go
from scipy import interpolate

In [11]:
PROJECT_ROOT = Path(os.getcwd()).resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
RAW_DIR = PROJECT_ROOT / "data/raw/stathead/annual_av"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
FIGURES_DIR = PROJECT_ROOT / "outputs/figures"
_TRADE_COLORS = ["#e15759", "#f28e2b", "#76b7b2", "#59a14f", "#b07aa1"]

In [21]:
PFF_WAR = pl.read_csv(PROCESSED_DIR / "pff_war_draft_chart.csv", infer_schema_length=None)
f = interpolate.interp1d(PFF_WAR["Pick"], PFF_WAR["PFF_WAR_Normalized"])
Pick = list(range(1,225))
PFF_WAR_Normalized = f(Pick)

In [25]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=Pick,
    y=PFF_WAR_Normalized,
    mode="lines",
))

fig.update_layout(
        title="Pick Value Comparison — Normalized to Pick 1",
        xaxis=dict(title="Overall Pick", range=[1, 225]),
        yaxis=dict(title="Normalized Value (Pick 1 = 1.0)"),
        template="plotly_white",
        legend=dict(groupclick="toggleitem"),
    )

fig.show()

In [27]:
df = pl.DataFrame({
    "Pick":Pick,
    "PFF_WAR_Normalized":PFF_WAR_Normalized
    })
df.write_csv("pff_war_draft_chart.csv")

In [40]:
trade_charts = {
    "Jimmy Johnson": (
        pl.read_csv(PROCESSED_DIR / "jimmy_johnson_trade_chart.csv", infer_schema_length=None)
        .unique(subset=["Pick"], keep="first")
        .sort("Pick")
        .select(["Pick", "Value"])
        .with_columns(pl.col("Pick").cast(pl.Int64))
    ),
    "Fitzgerald-Spielberger": (
        pl.read_csv(PROCESSED_DIR / "fitzgerald_spielberger_trade_chart.csv", infer_schema_length=None)
        .select(["Pick", "Value"])
        .with_columns(pl.col("Pick").cast(pl.Int64))
    ),
    "PFF WAR": (
        pl.read_csv(PROCESSED_DIR / "pff_war_draft_chart.csv", infer_schema_length=None)
        .rename({"PFF_WAR_Normalized": "Value"})
        .select(["Pick", "Value"])
        .with_columns(pl.col("Pick").cast(pl.Int64))
    ),
    "5-Year AV": (
        pl.read_csv(PROCESSED_DIR / "5_year_av_chart.csv", infer_schema_length=None)
        .rename({"Pk": "Pick", "FP Val": "Value"})
        .select(["Pick", "Value"])
        .with_columns(pl.col("Pick").cast(pl.Int64))
    ),
    "Rich Hill": (
        pl.read_csv(PROCESSED_DIR / "Rich-Hill.csv", infer_schema_length=None)
        .rename({"pick": "Pick", "value": "Value"})
        .select(["Pick", "Value"])
        .with_columns(pl.col("Pick").cast(pl.Int64))
    ),
}

In [41]:
fig = go.Figure()

for i, (label, df) in enumerate(trade_charts.items()):
        print(label)
        color = _TRADE_COLORS[i % len(_TRADE_COLORS)]
        val_at_1 = df.filter(pl.col("Pick") == 1)["Value"][0]
        chart_df = df.filter(pl.col("Pick") <= 250)
        x = chart_df["Pick"].to_list()
        y_norm = [v / val_at_1 for v in chart_df["Value"].to_list()]

        fig.add_trace(go.Scatter(
            x=x,
            y=y_norm,
            mode="lines",
            line=dict(color=color, width=2, dash="dash"),
            name=label,
            legendgroup="trade_charts",
            legendgrouptitle=dict(text="Trade Charts") if i == 0 else {},
        ))

fig.update_layout(
        title="Pick Value Comparison — Normalized to Pick 1",
        xaxis=dict(title="Overall Pick", range=[1, 250]),
        yaxis=dict(title="Normalized Value (Pick 1 = 1.0)"),
        template="plotly_white",
        legend=dict(groupclick="toggleitem"),
    )

fig.show()

Jimmy Johnson
Fitzgerald-Spielberger
PFF WAR
5-Year AV
Rich Hill


In [2]:
# Load current season play-by-play data
pbp = nfl.load_pbp()

# Load player game-level stats for multiple seasons
player_stats = nfl.load_player_stats([2022, 2023])

In [6]:
pfr_advanced = nfl.load_pfr_advstats([2022, 2023])

In [7]:
pfr_advanced.columns

['game_id',
 'pfr_game_id',
 'season',
 'week',
 'game_type',
 'team',
 'opponent',
 'pfr_player_name',
 'pfr_player_id',
 'passing_drops',
 'passing_drop_pct',
 'receiving_drop',
 'receiving_drop_pct',
 'passing_bad_throws',
 'passing_bad_throw_pct',
 'times_sacked',
 'times_blitzed',
 'times_hurried',
 'times_hit',
 'times_pressured',
 'times_pressured_pct',
 'def_times_blitzed',
 'def_times_hurried',
 'def_times_hitqb']

In [8]:
import pandas as pd